# Online Shoppers — améliorer **precision ET recall** avec notre arbre from scratch

**Objectif de l'enseignant :** obtenir un bon compromis entre précision et rappel, sans abandonner l'algorithme implémenté nous-mêmes.

Ce notebook est autonome pour **Google Colab** : il ne nécessite pas de dossier `src`. Il suit notre protocole 60/20/20 (seed 42), utilise les 16 variables originales (sans `PageValues`), implémente Gini pondéré, construit l’arbre par recherche de splits, vérifie des tests, compare scikit-learn, recherche les poids/profondeurs/tailles de feuilles/**seuils de décision**, puis analyse les confusions.

**Règle scientifique :** sélectionner tout uniquement sur la validation. Le test est évalué à la fin. Les objectifs **precision ≥ 0.50, recall ≥ 0.50** sont des *cibles exploratoires*, pas des résultats promis. Notre ancien test a déjà été observé : pour une évaluation réellement indépendante après ces nouveaux choix, il faudrait un nouveau jeu de test externe ou un protocole d'évaluation imbriqué.

**Exécution :** Colab → *Runtime → Run all* → importer `online_shoppers_intention.csv`. Le calcul des arbres de zéro peut prendre quelques minutes.

## 1 — Dataset, EDA et classes

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier
from google.colab import files
from IPython.display import display

# Getting data from online_shoppers_intention
uploaded = files.upload()
csv_files = [name for name in uploaded if name.lower().endswith('.csv')]
if not csv_files:
    raise ValueError('Please upload online_shoppers_intention.csv')
data = pd.read_csv(csv_files[0])

print('Dataset shape:', data.shape)
print('Missing values:', int(data.isna().sum().sum()))
print('Duplicated rows:', int(data.duplicated().sum()))
print('Purchases and non-purchases:')
print(data['Revenue'].value_counts())
print('Purchase rate:', round(data['Revenue'].astype(int).mean() * 100, 2), '%')
assert 'Revenue' in data.columns

In [ ]:
# Bar chart of the target distribution
counts = data['Revenue'].astype(int).value_counts().sort_index()
plt.figure(figsize=(6, 4))
plt.bar(['No purchase (0)', 'Purchase (1)'], [counts[0], counts[1]])
plt.title('Target distribution')
plt.ylabel('Number of shopping sessions')
plt.show()

## 2 — Mélange reproductible, train / validation / test, one-hot encoding

In [ ]:
# Our input variables
features = [
    'Administrative', 'Administrative_Duration',
    'Informational', 'Informational_Duration',
    'ProductRelated', 'ProductRelated_Duration',
    'BounceRates', 'ExitRates', 'SpecialDay',
    'Month', 'OperatingSystems', 'Browser',
    'Region', 'TrafficType', 'VisitorType', 'Weekend'
]
categories = [
    'Month', 'OperatingSystems', 'Browser', 'Region',
    'TrafficType', 'VisitorType', 'Weekend'
]

X = data[features]
y = data['Revenue'].astype(int)

# Shuffle the row indices
n = len(data)
indices = np.random.default_rng(42).permutation(n)
n_test = round(0.2 * n)
n_val = round(0.2 * n)
test_idx = indices[:n_test]
val_idx = indices[n_test:n_test + n_val]
train_idx = indices[n_test + n_val:]

X_train = X.iloc[train_idx].copy()
X_val = X.iloc[val_idx].copy()
X_test = X.iloc[test_idx].copy()
y_train = y.iloc[train_idx].to_numpy()
y_val = y.iloc[val_idx].to_numpy()
y_test = y.iloc[test_idx].to_numpy()

# Convert categories to text
for column in categories:
    X_train[column] = X_train[column].astype(str)
    X_val[column] = X_val[column].astype(str)
    X_test[column] = X_test[column].astype(str)

# Convert categories into numerical columns
train_encoded = pd.get_dummies(X_train, columns=categories, dtype=float)
val_encoded = pd.get_dummies(X_val, columns=categories, dtype=float)
test_encoded = pd.get_dummies(X_test, columns=categories, dtype=float)
val_encoded = val_encoded.reindex(columns=train_encoded.columns, fill_value=0.0)
test_encoded = test_encoded.reindex(columns=train_encoded.columns, fill_value=0.0)
X_train_ready = train_encoded.to_numpy(dtype=float)
X_val_ready = val_encoded.to_numpy(dtype=float)
X_test_ready = test_encoded.to_numpy(dtype=float)

print('Training shape:', X_train_ready.shape)
print('Validation shape:', X_val_ready.shape)
print('Test shape:', X_test_ready.shape)
print('Purchase rates:', *(f'{z.mean()*100:.2f}%' for z in (y_train, y_val, y_test)))
assert X_train_ready.shape[1] == X_val_ready.shape[1] == X_test_ready.shape[1]
assert np.isfinite(X_train_ready).all()

In [ ]:
# Comprendre si la proportion des achats reste similaire dans chaque split
sizes = [len(y_train), len(y_val), len(y_test)]
labels = ['Training', 'Validation', 'Test']
fig, axs = plt.subplots(1, 2, figsize=(12, 4))
axs[0].bar(labels, sizes)
axs[0].set(title='Taille des trois ensembles', ylabel='Sessions')
axs[1].bar(labels, [np.mean(z)*100 for z in [y_train, y_val, y_test]])
axs[1].set(title='Part des achats par ensemble', ylabel='Achats (%)', ylim=(0,25))
plt.tight_layout(); plt.show()
assert len(y_train)==7398 and len(y_val)==2466 and len(y_test)==2466


## 3 — Notre algorithme, entièrement from scratch

On utilise l’impureté de Gini pondérée et la récursion. **Amélioration importante** : les feuilles conservent une *fraction pondérée* d'achats, pour permettre un seuil de décision ajustable après entraînement. Le seuil `0.5` donne exactement la prédiction de majorité pondérée de notre version précédente.

Attention : il s'agit d'un **score pondéré**, pas nécessairement d'une probabilité calibrée d'achat.

In [ ]:
# Calculating Gini impurity
def gini(y, sample_weight=None):
    y = np.asarray(y)
    if len(y) == 0:
        return 0.0
    if sample_weight is None:
        sample_weight = np.ones(len(y))
    total_weight = np.sum(sample_weight)
    if total_weight == 0:
        return 0.0
    p = np.sum(sample_weight[y == 1]) / total_weight
    return 1 - p**2 - (1 - p)**2

# Calculating weighted Gini impurity
def weighted_gini(y_left, y_right, w_left=None, w_right=None):
    if w_left is None:
        w_left = np.ones(len(y_left))
    if w_right is None:
        w_right = np.ones(len(y_right))
    total_left = np.sum(w_left)
    total_right = np.sum(w_right)
    total = total_left + total_right
    if total == 0:
        return 0.0
    return (total_left / total) * gini(y_left, w_left) + (total_right / total) * gini(y_right, w_right)

# Finding the best feature and threshold
def best_split(X, y, min_samples_leaf=20, sample_weight=None):
    if sample_weight is None:
        sample_weight = np.ones(len(y))
    best_feature = None
    best_threshold = None
    best_score = gini(y, sample_weight)
    for j in range(X.shape[1]):
        values = np.unique(X[:, j])
        if len(values) <= 1:
            continue
        if len(values) <= 16:
            thresholds = (values[:-1] + values[1:]) / 2
        else:
            thresholds = np.unique(np.quantile(values, np.linspace(0, 1, 17)[1:-1]))
        for t in thresholds:
            left = X[:, j] <= t
            right = ~left
            if left.sum() < min_samples_leaf or right.sum() < min_samples_leaf:
                continue
            score = weighted_gini(y[left], y[right], sample_weight[left], sample_weight[right])
            if score < best_score - 1e-12:
                best_score = score
                best_feature = j
                best_threshold = t
    return best_feature, best_threshold

# Building the decision tree
def build_tree(X, y, depth=0, max_depth=5, min_samples_leaf=20, sample_weight=None):
    if sample_weight is None:
        sample_weight = np.ones(len(y))
    weight_0 = np.sum(sample_weight[y == 0])
    weight_1 = np.sum(sample_weight[y == 1])
    prediction = int(weight_1 >= weight_0)
    score = weight_1 / (weight_0 + weight_1) if (weight_0 + weight_1) > 0 else 0.0
    if depth >= max_depth or gini(y, sample_weight) <= 1e-12:
        return {'prediction': prediction, 'score': score}
    j, t = best_split(X, y, min_samples_leaf=min_samples_leaf, sample_weight=sample_weight)
    if j is None:
        return {'prediction': prediction, 'score': score}
    left = X[:, j] <= t
    right = ~left
    return {
        'feature': j, 'threshold': t,
        'left': build_tree(X[left], y[left], depth+1, max_depth, min_samples_leaf, sample_weight[left]),
        'right': build_tree(X[right], y[right], depth+1, max_depth, min_samples_leaf, sample_weight[right])
    }

# Predicting one shopping session
def predict_one(x, tree):
    if 'prediction' in tree:
        return tree['prediction']
    if x[tree['feature']] <= tree['threshold']:
        return predict_one(x, tree['left'])
    return predict_one(x, tree['right'])

# Predicting all shopping sessions
def predict(X, tree):
    return np.array([predict_one(x, tree) for x in X], dtype=int)

# Weighted score at the leaf (the decision cutoff is tuned on validation)
def score_one(x, tree):
    if 'score' in tree:
        return tree['score']
    if x[tree['feature']] <= tree['threshold']:
        return score_one(x, tree['left'])
    return score_one(x, tree['right'])

def predict_scores(X, tree):
    return np.array([score_one(x, tree) for x in X], dtype=float)


In [ ]:
# Checkpoints
assert np.isclose(gini(np.array([0, 0, 0, 0])), 0.0)
assert np.isclose(gini(np.array([0, 0, 1, 1])), 0.5)
assert np.isclose(gini(np.array([0, 0, 0, 0, 1])), 0.32)
assert np.isclose(weighted_gini(np.array([0, 0]), np.array([1, 1])), 0.0)
assert np.isclose(weighted_gini(np.array([0, 1]), np.array([0, 1])), 0.5)
assert np.isclose(weighted_gini(np.array([0, 0, 1]), np.array([1])), 1 / 3)
X_small = np.array([[1], [2], [3], [4]], dtype=float)
y_small = np.array([0, 0, 1, 1])
j, t = best_split(X_small, y_small, min_samples_leaf=1)
assert j == 0 and np.isclose(t, 2.5)
small_tree = build_tree(X_small, y_small, max_depth=2, min_samples_leaf=1)
assert np.array_equal(predict(X_small, small_tree), y_small)
print('OK - Gini, weighted Gini, best split, and tree predictions')

## 4 — Métriques : précision, recall, F1 et accuracy

- Precision = TP / (TP + FP) : quand on annonce *Purchase*, à quelle fréquence a-t-on raison ?
- Recall = TP / (TP + FN) : parmi les vrais achats, combien en détecte-t-on ?
- F1 : moyenne harmonique des deux.

**Notre objectif** : trouver sur la validation un modèle qui satisfait **precision ≥ 0.50 ET recall ≥ 0.50**. Si aucun candidat ne respecte ces conditions, le notebook le dira clairement et choisira le meilleur F1 de validation comme solution de repli.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

def classification_metrics(y, pred):
    y = np.asarray(y); pred = np.asarray(pred)
    tp = np.sum((y == 1) & (pred == 1))
    fp = np.sum((y == 0) & (pred == 1))
    fn = np.sum((y == 1) & (pred == 0))
    precision = tp / (tp + fp) if tp+fp else 0.0
    recall = tp / (tp + fn) if tp+fn else 0.0
    f1 = 2*precision*recall/(precision+recall) if precision+recall else 0.0
    return {'Accuracy': np.mean(y == pred), 'Precision': precision,
            'Recall': recall, 'F1': f1}

TARGET_PRECISION = 0.50
TARGET_RECALL = 0.50
print('Cibles exploratoires :', TARGET_PRECISION, TARGET_RECALL)


## 5 — Recherche sur validation : profondeur + feuilles + pondération + seuil

On fait varier :
- profondeur : **3, 5, 7** ;
- `min_samples_leaf` : **20, 50** ;
- poids des achats : **1, 2, 3, 4, ratio ≈ 5.57** ;
- seuil du score de feuille : **0.20 à 0.80**.

Cette recherche reste volontairement limitée pour Colab. Elle comprend **30 arbres from scratch** ; chaque arbre est ensuite évalué à différents seuils sans être ré-entraîné. On vérifie également une version scikit-learn comparable (même poids et complexité), dont la recherche exacte des splits peut lui permettre de différer de notre version.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

purchase_ratio = float(np.sum(y_train == 0)/np.sum(y_train == 1))
weights_to_try = [1.0, 2.0, 3.0, 4.0, purchase_ratio]
thresholds_to_try = np.round(np.arange(0.20, 0.801, 0.05), 2)
records = []
trained_models = {}
for depth in [3, 5, 7]:
    for min_leaf in [20, 50]:
        for weight in weights_to_try:
            sw = np.where(y_train == 1, weight, 1.0)
            tree = build_tree(X_train_ready, y_train, max_depth=depth,
                              min_samples_leaf=min_leaf, sample_weight=sw)
            val_scores = predict_scores(X_val_ready, tree)
            trained_models[('Scratch', depth, min_leaf, weight)] = tree
            for threshold in thresholds_to_try:
                metrics = classification_metrics(y_val, (val_scores >= threshold).astype(int))
                records.append({'Method':'Scratch', 'Depth':depth, 'Min_leaf':min_leaf,
                                'Purchase_weight':weight, 'Threshold':threshold, **metrics})
            # Same training sample and weight for the library baseline
            sk = DecisionTreeClassifier(max_depth=depth, min_samples_leaf=min_leaf,
                                        criterion='gini', random_state=42)
            sk.fit(X_train_ready, y_train, sample_weight=sw)
            sk_scores = sk.predict_proba(X_val_ready)[:, list(sk.classes_).index(1)]
            trained_models[('Sklearn', depth, min_leaf, weight)] = sk
            for threshold in thresholds_to_try:
                metrics = classification_metrics(y_val, (sk_scores >= threshold).astype(int))
                records.append({'Method':'Sklearn', 'Depth':depth, 'Min_leaf':min_leaf,
                                'Purchase_weight':weight, 'Threshold':threshold, **metrics})

validation_table = pd.DataFrame(records)
print('Configurations évaluées :', len(validation_table))
print('Meilleures configurations Scratch par F1 (VALIDATION) :')
display(validation_table[validation_table.Method=='Scratch'].sort_values('F1',ascending=False).head(12).round(4))


## 6 — Sélection : **les deux objectifs** avant le F1

On préfère les configurations qui respectent **precision ≥ 0.50 ET recall ≥ 0.50** sur validation ; parmi celles-ci, on retient le F1 le plus élevé. Si aucune ne réussit les deux contraintes, nous le montrons et retenons le meilleur F1 en indiquant explicitement qu'il ne satisfait pas le cahier des charges.

On sélectionne **séparément** le meilleur modèle from scratch et le meilleur benchmark scikit-learn. Le benchmark ne remplace pas notre travail personnel.

In [ ]:
def choose_model(method):
    subset = validation_table[validation_table['Method'] == method].copy()
    eligible = subset[(subset['Precision'] >= TARGET_PRECISION) &
                      (subset['Recall'] >= TARGET_RECALL)]
    passes = not eligible.empty
    chosen = (eligible if passes else subset).sort_values(
        ['F1', 'Precision', 'Recall'], ascending=False).iloc[0]
    return chosen, passes

scratch_chosen, scratch_passes = choose_model('Scratch')
sk_chosen, sk_passes = choose_model('Sklearn')
for row, passed in [(scratch_chosen,scratch_passes),(sk_chosen,sk_passes)]:
    print(row['Method'], '— objectifs validation atteints ? ', 'OUI' if passed else 'NON')
    print(row[['Depth','Min_leaf','Purchase_weight','Threshold','Accuracy','Precision','Recall','F1']].round(4).to_string())
    print()


In [ ]:
# Pareto-style scatter for all Scratch candidates on VALIDATION
v = validation_table[validation_table.Method=='Scratch']
fig, ax=plt.subplots(figsize=(8,6))
sc = ax.scatter(v['Recall'],v['Precision'],c=v['F1'],cmap='viridis', alpha=0.55, s=22)
ax.axvline(TARGET_RECALL,color='tab:red',linestyle='--',label='Recall target')
ax.axhline(TARGET_PRECISION,color='tab:orange',linestyle='--',label='Precision target')
ax.scatter([scratch_chosen['Recall']],[scratch_chosen['Precision']],marker='*',s=230,
           edgecolor='black',color='gold',label='Selected from scratch')
ax.set(xlim=(0,1),ylim=(0,1),xlabel='Recall',ylabel='Precision',
       title='VALIDATION — Precision/Recall trade-off (Scratch)')
ax.legend();fig.colorbar(sc,ax=ax,label='F1');plt.tight_layout();plt.show()


## 7 — Test final des modèles sélectionnés

**Ne changez plus les hyperparamètres en fonction de ce tableau.** Les données de test sont ici uniquement utilisées pour une évaluation descriptive. Notre projet a déjà montré certains résultats du même test, donc toute nouvelle estimation prétendant être totalement indépendante demanderait un nouveau jeu externe ou un protocole imbriqué.

On conserve également **un arbre pondéré de référence (depth 5, leaf 20, weight ratio, threshold 0.5)** pour vérifier la cohérence avec le résultat historique : test F1 ≈ **0.4214**.

In [ ]:
def fitted_test_prediction(chosen):
    key = (chosen['Method'], int(chosen['Depth']), int(chosen['Min_leaf']),
           float(chosen['Purchase_weight']))
    model=trained_models[key]
    if chosen['Method']=='Scratch':
        score=predict_scores(X_test_ready,model)
    else:
        score=model.predict_proba(X_test_ready)[:,list(model.classes_).index(1)]
    return (score >= float(chosen['Threshold'])).astype(int)

selected_predictions={
    'Baseline No Purchase':np.zeros(len(y_test),dtype=int),
    'Selected scratch tree':fitted_test_prediction(scratch_chosen),
    'Selected sklearn tree':fitted_test_prediction(sk_chosen)
}
ref_key=('Scratch',5,20,purchase_ratio)
if ref_key in trained_models:
    selected_predictions['Previous weighted reference']=(predict_scores(X_test_ready,trained_models[ref_key])>=0.5).astype(int)

final_results=[]
for name,pred in selected_predictions.items():
    tn,fp,fn,tp=confusion_matrix(y_test,pred,labels=[0,1]).ravel()
    final_results.append({'Model':name,**classification_metrics(y_test,pred),
                          'TN':tn,'FP':fp,'FN':fn,'TP':tp})
final_df=pd.DataFrame(final_results)
display(final_df.round(4))
print('La comparaison principale est sélectionnée sur VALIDATION ; les chiffres ci-dessus sont des TEST.')

fig,axes=plt.subplots(1,2,figsize=(11,4))
for ax,name in zip(axes,['Selected scratch tree','Selected sklearn tree']):
    ConfusionMatrixDisplay(confusion_matrix(y_test,selected_predictions[name],labels=[0,1]),
                           display_labels=['No purchase','Purchase']).plot(ax=ax,colorbar=False)
    ax.set_title(name)
plt.tight_layout();plt.show()


## 8 — Conclusion automatique : qu'avons-nous amélioré ?

La réponse doit être fondée sur les résultats **mesurés**, sans annoncer de réussite à l'avance. Un F1 plus élevé n'implique pas automatiquement que les deux métriques de la cible sont meilleures. Rappel des résultats *historiques* sur ce test : arbre original precision **0.4000**, recall **0.1055**, F1 **0.1670** ; arbre pondéré precision **0.3132**, recall **0.6438**, F1 **0.4214**.

In [ ]:
chosen_test = final_df.loc[final_df['Model']=='Selected scratch tree'].iloc[0]
print('CONCLUSION')
print('Precision du nouvel arbre :',f"{chosen_test['Precision']:.4f}")
print('Recall du nouvel arbre :',f"{chosen_test['Recall']:.4f}")
print('F1 du nouvel arbre :',f"{chosen_test['F1']:.4f}")
if scratch_passes:
    print('Les deux objectifs ont été atteints sur VALIDATION ; vérifier la généralisation sur test.')
else:
    print('Aucun candidat n’a atteint simultanément les deux objectifs sur VALIDATION.')
if chosen_test['Precision']>=TARGET_PRECISION and chosen_test['Recall']>=TARGET_RECALL:
    print('Les deux seuils sont atteints sur le test observé (à confirmer indépendamment).')
else:
    print('Sur ce test, au moins un objectif precision/recall reste non atteint.')
print('Amélioration F1 vs ancienne version pondérée (0.4214) :',
      f"{chosen_test['F1']-0.4214:+.4f}")


## 9 — Vérifications des consignes du professeur et limites

✅ Arbre de classification **codé de zéro** (Gini, score de split, récursion, inférence et seuil).

✅ Comparaison à **DecisionTreeClassifier** de scikit-learn sur les mêmes jeux de données.

✅ **Train / validation / test** reproductibles, contrôle des shapes, aucune sélection de paramètres sur le test.

✅ **Precision, recall, F1, accuracy, matrices de confusion**, visualisation du compromis, discussion critique.

✅ Analyse de **l'imbalance**, recherche de pondérations, tailles de feuille, profondeurs et seuils.

⚠️ Le modèle de zéro conserve la recherche quantile des seuils pour la vitesse, différente de la recherche exhaustive scikit-learn.

⚠️ Le dataset original contient **125 doublons**. Les splits aléatoires peuvent placer des sessions identiques dans des ensembles différents. Prévoir un contrôle de sensibilité avec dédoublonnage pour l'étude finale.

⚠️ Les objectifs 50%/50% ne sont pas garantis ; si aucun modèle ne les atteint sur validation, il faut le signaler et envisager un autre dataset **avec une justification scientifique**.

⚠️ Comme le test d’origine a déjà été examiné dans les expériences précédentes, une nouvelle évaluation non biaisée demanderait idéalement un autre test externe ou une validation croisée imbriquée.

**Question de recherche finale :** *Dans quelle mesure l’optimisation des règles de split, de la complexité de l'arbre, du poids des classes et du seuil de décision permet-elle d'améliorer simultanément précision et rappel pour prédire les achats en ligne ?*